# Session 13 prep — decision threshold for the at-application model

The plan (buổi 13) says the Streamlit form should show "Duyệt/Từ chối kèm xác suất rủi ro"
but never says what probability counts as "reject". This notebook derives that threshold
from real portfolio economics instead of using the default 0.5, and documents why 0.5 -
and two other common "principled-looking" choices - are each the wrong tool for this
specific model.

**Held-out set**: the exact session 9-11 test split (`train_test_split(..., test_size=0.2,
random_state=42, stratify=...)` on `ml_features`) - 6,517 rows never used to fit
`models/at_application_model.pkl`, the same split session 9-11 report AUC/PR-AUC against.

In [1]:
import sys
sys.path.insert(0, '../etl')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_curve, f1_score, precision_score, recall_score, average_precision_score,
)

from config import get_engine
from model.features import TARGET_COL

engine = get_engine()
df = pd.read_sql("SELECT * FROM ml_features", engine)
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df[TARGET_COL]
)

bundle = joblib.load("../models/at_application_model.pkl")
X_test = test_df[bundle["feature_names"]]
y_test = test_df[TARGET_COL].to_numpy()
proba = bundle["model"].predict_proba(bundle["preprocessor"].transform(X_test))[:, 1]

print(f"test set: {len(test_df)} rows, default rate {y_test.mean():.4f}")
print(f"bundle metrics (for cross-check): {bundle['metrics']['roc_auc']:.4f} ROC-AUC CV mean")

test set: 6517 rows, default rate 0.2182
bundle metrics (for cross-check): 0.8928 ROC-AUC CV mean


## Step 1 — is `predict_proba` actually a calibrated probability?

This has to be checked before using any formula that treats the model's output as
"P(default)" in the literal sense. `build_pipeline` (session 10) sets `scale_pos_weight`
on XGBoost specifically to correct for the ~21.8% base rate - which reweights the loss
function and, as a direct consequence, shifts predicted scores away from true
probabilities. If that shift is real, a theoretical threshold formula applied to the raw
scores will recommend the wrong cutoff, and only a check against real outcomes catches it.

In [2]:
print(f"actual default rate in test set : {y_test.mean():.4f}")
print(f"mean predicted P(default)       : {proba.mean():.4f}")

calib = pd.DataFrame({"proba": proba, "y": y_test})
calib["bin"] = pd.qcut(calib["proba"], 10, duplicates="drop")
calib_summary = calib.groupby("bin", observed=True).agg(
    mean_predicted=("proba", "mean"), actual_default_rate=("y", "mean"), n=("y", "size")
)
calib_summary

actual default rate in test set : 0.2182
mean predicted P(default)       : 0.3644


,mean_predicted,actual_default_rate,n
bin,,,
"(0.0018800000000000002, 0.0703]",0.037410,0.000000,652
"(0.0703, 0.131]",0.102515,0.016871,652
"(0.131, 0.181]",0.155114,0.023041,651
"(0.181, 0.233]",0.206656,0.041411,652
"(0.233, 0.292]",0.261617,0.064417,652
"(0.292, 0.357]",0.324714,0.089094,651
"(0.357, 0.435]",0.394813,0.141104,652
"(0.435, 0.564]",0.498825,0.244240,651
"(0.564, 0.855]",0.683632,0.564417,652


**Result: badly miscalibrated.** Mean predicted P(default) is 0.36 against an actual
base rate of 0.22, and the bin table shows it's not a uniform shift: at a predicted score
of ~0.50, the true default rate in that bin is only ~24%, not 50%. A closed-form formula
like `t* = C_FP / (C_FP + C_FN)` assumes `proba` means what it says - it doesn't here, so
that formula would recommend the wrong number even with the right costs plugged in. The
rest of this notebook never treats `proba` as a probability - only as a ranking score,
and finds the cutoff by measuring real outcomes directly.

## Step 2 — real cost of each error type

Two numbers, both computed from real columns in the held-out set (not model inputs -
`loan_int_rate` never reaches the model; it's only used here, after the fact, to price a
historical outcome whose true rate is already known):

- **Cost of approving a defaulter (false negative)**: the loan's principal, assuming a
  stated recovery rate on default. `recovery_rate=0` is the conservative case (unsecured
  personal loan, no collateral) - checked for sensitivity below.
- **Cost of rejecting a good applicant (false positive)**: the interest income the lender
  would have earned, approximated as simple (non-compounded) interest over the loan term -
  a deliberate simplification, not a full NPV model.

In [3]:
loan_amnt = test_df["loan_amnt"].to_numpy()
loan_int_rate = test_df["loan_int_rate"].to_numpy()
term_years = test_df["loan_term_months"].to_numpy() / 12

print(f"mean loan_amnt     : {loan_amnt.mean():,.0f}")
print(f"mean loan_int_rate : {loan_int_rate.mean():.2f}%")
print(f"mean term          : {term_years.mean():.2f} years")
print()
print(f"illustrative C_FN (recovery=0%) : {loan_amnt.mean():,.0f}")
print(f"illustrative C_FP               : {(loan_amnt * loan_int_rate/100 * term_years).mean():,.0f}")
print()
naive_bayes_threshold = (loan_amnt * loan_int_rate/100 * term_years).mean() / (
    (loan_amnt * loan_int_rate/100 * term_years).mean() + loan_amnt.mean()
)
print(f"theoretical Bayes threshold if proba WERE calibrated: {naive_bayes_threshold:.3f}")
print("(kept only to show how wrong it would be - see step 1)")

mean loan_amnt     : 9,670
mean loan_int_rate : 11.00%
mean term          : 3.21 years

illustrative C_FN (recovery=0%) : 9,670
illustrative C_FP               : 3,515

theoretical Bayes threshold if proba WERE calibrated: 0.267
(kept only to show how wrong it would be - see step 1)


## Step 3 — empirical cost-minimizing threshold

Sweep every threshold on the held-out set, compute total real cost at each using the
actual labels and the actual `loan_amnt`/`loan_int_rate`/`loan_term_months` of every row,
and pick the minimum. This never assumes `proba` is calibrated - it only uses it to rank
applicants, then measures the real-dollar consequence of the cutoff against real
outcomes.

In [4]:
def total_cost(threshold, recovery_rate):
    predicted_reject = proba >= threshold
    fn_mask = (y_test == 1) & (~predicted_reject)   # approved, actually defaulted
    fp_mask = (y_test == 0) & (predicted_reject)     # rejected, would have repaid
    cost_fn = (loan_amnt[fn_mask] * (1 - recovery_rate)).sum()
    cost_fp = (loan_amnt[fp_mask] * (loan_int_rate[fp_mask] / 100) * term_years[fp_mask]).sum()
    return cost_fn, cost_fp, cost_fn + cost_fp

thresholds = np.linspace(0.01, 0.99, 197)

sensitivity = {}
for recovery_rate in (0.0, 0.2, 0.4):
    costs = [total_cost(t, recovery_rate)[2] for t in thresholds]
    best_t = thresholds[int(np.argmin(costs))]
    avg_cost = min(costs) / len(y_test)
    sensitivity[recovery_rate] = (best_t, avg_cost)
    print(f"recovery_rate={recovery_rate:.1f}: best threshold = {best_t:.3f} | "
          f"avg cost/applicant = {avg_cost:,.0f}")

RECOMMENDED_THRESHOLD = round(sensitivity[0.0][0], 2)
print()
print(f"RECOMMENDED_THRESHOLD = {RECOMMENDED_THRESHOLD} "
      "(conservative: recovery_rate=0%, and the number barely moves even at 40% recovery)")

recovery_rate=0.0: best threshold = 0.530 | avg cost/applicant = 721
recovery_rate=0.2: best threshold = 0.565 | avg cost/applicant = 623
recovery_rate=0.4: best threshold = 0.565 | avg cost/applicant = 510

RECOMMENDED_THRESHOLD = 0.53 (conservative: recovery_rate=0%, and the number barely moves even at 40% recovery)


## Step 4 — comparison against other common threshold choices

Same held-out set, same model, four alternative rules that all *look* principled but
answer a different question than "what does this cost the lender":

In [5]:
def evaluate(t, label):
    pred = proba >= t
    fn_mask = (y_test == 1) & (~pred)
    fp_mask = (y_test == 0) & pred
    cost = (
        (loan_amnt[fn_mask] * 1.0).sum()
        + (loan_amnt[fp_mask] * (loan_int_rate[fp_mask] / 100) * term_years[fp_mask]).sum()
    )
    return {
        "threshold": round(t, 3),
        "approval_rate": round((~pred).mean(), 3),
        "precision": round(precision_score(y_test, pred, zero_division=0), 3),
        "recall": round(recall_score(y_test, pred, zero_division=0), 3),
        "f1": round(f1_score(y_test, pred, zero_division=0), 3),
        "avg_cost_per_applicant": round(cost / len(y_test)),
    }

results = {}
results["naive_0.5"] = evaluate(0.5, "naive 0.5")

fpr, tpr, roc_thresh = roc_curve(y_test, proba)
youden_t = roc_thresh[np.argmax(tpr - fpr)]
results["youden_j"] = evaluate(youden_t, "Youden's J")

f1s = [f1_score(y_test, proba >= t, zero_division=0) for t in thresholds]
f1_t = thresholds[int(np.argmax(f1s))]
results["f1_max"] = evaluate(f1_t, "F1-max")

target_recall = 0.90
candidates = [t for t in thresholds if recall_score(y_test, proba >= t, zero_division=0) >= target_recall]
recall_t = max(candidates) if candidates else thresholds[0]
results["recall_target_90pct"] = evaluate(recall_t, "recall>=90%")

results["cost_minimizing"] = evaluate(RECOMMENDED_THRESHOLD, "cost-minimizing (recommended)")

comparison = pd.DataFrame(results).T
comparison

,threshold,approval_rate,precision,recall,f1,avg_cost_per_applicant
naive_0.5,0.500,0.751,0.677,0.773,0.722,743.0
youden_j,0.441,0.704,0.609,0.826,0.701,761.0
f1_max,0.565,0.801,0.782,0.715,0.747,736.0
recall_target_90pct,0.340,0.574,0.464,0.904,0.613,1006.0
cost_minimizing,0.530,0.774,0.727,0.754,0.740,721.0


## Decision

**Recommended: threshold = 0.53.** `probability >= 0.53` -> reject, `< 0.53` -> approve.

- Lowest average real cost per applicant among every option tested (see table above).
- Stable under the one real assumption behind it (recovery rate on default): 0.0 -> 0.53,
  0.2 and 0.4 -> 0.565. The recommendation doesn't hinge on guessing that number exactly
  right.
- Every other option either treats the two error types as equally costly (Youden's J,
  F1-max — a false rejection and a bad loan approval are not the same size of mistake),
  or optimizes a policy target instead of cost (recall>=90%, which produced the *highest*
  cost of all five options here by rejecting far more good applicants than the economics
  justify).
- The naive 0.5 default happens to land close to the cost-minimizing region on this
  particular dataset, but for a reason that doesn't generalize: it isn't calibrated
  either (step 1), so its apparent reasonableness here is a property of this dataset, not
  a property of the number 0.5.

**Caveats to carry forward**: recovery_rate=0% is a stated simplifying assumption, not
a measured one (this dataset has no collections/recovery data); the FP cost uses simple
interest with no discounting or cost-of-capital netting. Good enough for picking an
operating point for a portfolio demo, not a substitute for a real credit-risk cost model.
`loan_int_rate` is used here only to price historical outcomes after the fact - it never
reaches the model as an input, so this does not reopen the leakage question from session
8-9.